# Mining Operations Data Preparation & Feature Engineering

## Project Objective

Prepare the mining operations dataset for reliable downstream analysis and machine learning by validating data quality, integrating relevant tables, and engineering documented operational features.

## Scope

1. Load and validate the source data.
2. Perform data cleaning and consistency checks.
3. Integrate production, cost, and mine-site data (daily, per site).
4. Aggregate equipment and safety data to the same grain (site-day) and add them as features.
5. Create and document analytical features.
6. Validate, document (data dictionary), and export the prepared dataset.

## Data Source

Synthetic Mining Operations Database (`mining_operations.db`), previously audited in `01_Mining_Operations_Database_Audit.ipynb` and explored in `02_Mining_Operations_EDA.ipynb`.

## Expected Outcome

A validated, documented dataset with one row per mine site per day, plus a data dictionary, suitable for downstream analytical workflows.

## Notebook Structure

| Section | Topic |
|---|---|
| 1 | Environment and database connection |
| 2 | Source validation |
| 3 | Missing `automation_level` |
| 4 | Data integration |
| 5 | Cost reconciliation |
| 6 | Value and business-rule checks, zero-production records |
| 7 | Feature engineering |
| 8 | Prepared dataset and validation |
| 9 | Notes for downstream use |
| 10 | Data dictionary and export |
| 11 | Summary and conclusion |

> **Principle:** Source values are not modified. Every change made in the prepared dataset (renamed, dropped, or derived columns) is documented.

## 1. Environment and Database Connection

In [1]:
import os
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
from google.colab import drive

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 100)

# Mount Google Drive (safe to re-run)
drive.mount('/content/drive')

print("Libraries imported successfully.")

Mounted at /content/drive
Libraries imported successfully.


In [2]:
# Locate the SQLite database in Google Drive
db_candidates = [
    Path(root) / file
    for root, dirs, files in os.walk('/content/drive/MyDrive')
    for file in files
    if file.lower() == 'mining_operations.db'
]

print("Database files found:", len(db_candidates))
for path in db_candidates:
    print(path)

if not db_candidates:
    raise ValueError("Database not found. Check its location in Google Drive.")

if len(db_candidates) > 1:
    print("\nWarning: more than one file found. The first one is used.")

DB_PATH = db_candidates[0]
OUTPUT_DIR = DB_PATH.parent   # prepared files are saved next to the database

conn = sqlite3.connect(DB_PATH)

print("\nDatabase connection successful!")
print("Database path:", DB_PATH)
print("Output folder:", OUTPUT_DIR)

Database files found: 1
/content/drive/MyDrive/Mining-Operation/mining_operations.db

Database connection successful!
Database path: /content/drive/MyDrive/Mining-Operation/mining_operations.db
Output folder: /content/drive/MyDrive/Mining-Operation


## 2. Source Validation

Memastikan database yang dimuat sama dengan yang sudah diaudit (jumlah tabel dan baris), serta memeriksa nilai kosong dan duplikat per tabel. Audit mendalam ada di `01_Mining_Operations_Database_Audit.ipynb`.

In [3]:
EXPECTED_ROWS = {
    "daily_costs": 1825,
    "daily_production": 1825,
    "equipment_performance": 8030,
    "mine_sites": 5,
    "safety_incidents": 360,
    "safety_metrics_summary": 4,
}

def read_table(table_name):
    quoted = '"' + table_name.replace('"', '""') + '"'
    return pd.read_sql_query(f"SELECT * FROM {quoted}", conn)

available_tables = pd.read_sql_query("""
    SELECT name FROM sqlite_master
    WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
""", conn)["name"].tolist()

assert set(available_tables) == set(EXPECTED_ROWS), f"Unexpected tables: {available_tables}"

tables = {name: read_table(name) for name in EXPECTED_ROWS}

profile_rows = []
for name, df in tables.items():
    profile_rows.append({
        "table_name": name,
        "rows": len(df),
        "expected_rows": EXPECTED_ROWS[name],
        "rows_match": len(df) == EXPECTED_ROWS[name],
        "columns": df.shape[1],
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_rows": int(df.duplicated().sum()),
    })

source_profile = pd.DataFrame(profile_rows)
display(source_profile)

assert source_profile["rows_match"].all(), "Row counts differ from the audited database."
print("Total rows across all tables:", f"{source_profile['rows'].sum():,}")

,table_name,rows,expected_rows,rows_match,columns,missing_values,duplicate_rows
0,daily_costs,1825,1825,True,14,0,0
1,daily_production,1825,1825,True,16,0,0
2,equipment_performance,8030,8030,True,16,0,0
3,mine_sites,5,5,True,16,1,0
4,safety_incidents,360,360,True,11,0,0
5,safety_metrics_summary,4,4,True,7,0,0


Total rows across all tables: 12,049


**Interpretasi validasi sumber**

- Enam tabel dan 12.049 baris sesuai dengan database yang diaudit. Tidak ada baris duplikat.
- Satu-satunya nilai kosong ada di `mine_sites.automation_level` (dibahas di Section 3).
- Tabel `equipment_performance` (8.030 baris) dan `safety_incidents` (360 baris) memiliki grain yang berbeda dari produksi harian, sehingga tidak digabung langsung. Keduanya diagregasi per lokasi per hari di Section 7 supaya baris tidak menggandakan.

## 3. Missing `automation_level`

In [4]:
sites_df = tables["mine_sites"].copy()

display(
    sites_df[["site_id", "site_name", "commodity", "status", "workforce_count", "automation_level"]]
    .sort_values("site_id")
)

,site_id,site_name,commodity,status,workforce_count,automation_level
3,COPP001,Copper Hills Mine,Copper,Operational,280,Low
1,GOLD001,Golden Valley Mine,Gold,Operational,1200,Medium
0,IRON001,Iron Ridge Mine,Iron Ore,Operational,850,High
2,LITH001,Lithium Creek Mine,Lithium,Operational,450,Medium
4,NICK001,Nickel Point Mine,Nickel,Care & Maintenance,25,NaN


**Keputusan data cleaning**

- Empat lokasi berstatus *Operational* memiliki `automation_level`. Nickel Point Mine (*Care & Maintenance*, 25 tenaga kerja) kosong.
- Status *Care & Maintenance* tidak otomatis berarti tingkat otomasi nol, dan tidak ada bukti untuk menentukan nilai sebenarnya. Nilai kosong **tidak ditebak**.
- Pada data sumber nilai tetap kosong. Pada dataset analitis (Section 7) nilainya diberi label eksplisit `Unknown` dan ditandai dengan kolom `automation_level_missing`, sehingga model dan pivot tidak mengabaikan barisnya tanpa disadari.

## 4. Data Integration

Produksi dan biaya digabung dengan key `(date, site_id)`, lalu informasi lokasi ditambahkan dari `mine_sites`.

### Key validation

In [5]:
production_df = tables["daily_production"].copy()
costs_df = tables["daily_costs"].copy()

key = ["date", "site_id"]

key_validation = pd.DataFrame([
    {"check": "Duplicate production keys", "result": int(production_df.duplicated(subset=key).sum())},
    {"check": "Duplicate cost keys", "result": int(costs_df.duplicated(subset=key).sum())},
    {"check": "Production keys without matching cost",
     "result": int(len(set(zip(production_df["date"], production_df["site_id"]))
                       - set(zip(costs_df["date"], costs_df["site_id"]))))},
    {"check": "Cost keys without matching production",
     "result": int(len(set(zip(costs_df["date"], costs_df["site_id"]))
                       - set(zip(production_df["date"], production_df["site_id"]))))},
])
display(key_validation)

,check,result
0,Duplicate production keys,0
1,Duplicate cost keys,0
2,Production keys without matching cost,0
3,Cost keys without matching production,0


Kombinasi `date` dan `site_id` unik di kedua tabel dan cocok satu-satu, sehingga penggabungan tidak menggandakan atau menghilangkan baris.

### Redundant columns
Kolom yang muncul di lebih dari satu tabel dibandingkan lebih dulu. Jika identik, hanya satu salinan yang dipertahankan.

In [6]:
pc_check = production_df.merge(
    costs_df, on=key, how="inner", validate="one_to_one", suffixes=("_production", "_cost")
)
pc_site_check = production_df.merge(
    sites_df, on="site_id", how="left", validate="many_to_one", suffixes=("_production", "_site")
)

redundancy_check = pd.DataFrame([
    {"comparison": "site_name: production vs costs",
     "differing_rows": int((pc_check["site_name_production"] != pc_check["site_name_cost"]).sum())},
    {"comparison": "commodity: production vs costs",
     "differing_rows": int((pc_check["commodity_production"] != pc_check["commodity_cost"]).sum())},
    {"comparison": "site_name: production vs mine_sites",
     "differing_rows": int((pc_site_check["site_name_production"] != pc_site_check["site_name_site"]).sum())},
    {"comparison": "commodity: production vs mine_sites",
     "differing_rows": int((pc_site_check["commodity_production"] != pc_site_check["commodity_site"]).sum())},
    {"comparison": "status: production vs mine_sites",
     "differing_rows": int((pc_site_check["status_production"] != pc_site_check["status_site"]).sum())},
    {"comparison": "tonnes_produced (costs) vs tonnes_mined (production)",
     "differing_rows": int((pc_check["tonnes_produced"] != pc_check["tonnes_mined"]).sum())},
])
display(redundancy_check)

,comparison,differing_rows
0,site_name: production vs costs,0
1,commodity: production vs costs,0
2,site_name: production vs mine_sites,0
3,commodity: production vs mine_sites,0
4,status: production vs mine_sites,0
5,tonnes_produced (costs) vs tonnes_mined (production),0


**Interpretasi**

- `site_name`, `commodity`, dan `status` identik di semua tabel. Satu salinan dipertahankan, sehingga kolom ganda `site_name_production`, `site_name_cost`, `commodity_production`, `commodity_cost`, dan `status_site` pada versi sebelumnya tidak lagi diperlukan.
- `tonnes_produced` di `daily_costs` sama persis dengan `tonnes_mined` pada semua baris (sesuai temuan audit), sehingga kolom ini dibuang dari dataset analitis sebagai duplikat.

### Merge production, costs, and mine sites

In [7]:
costs_slim = costs_df.drop(columns=["site_name", "commodity", "tonnes_produced"])

production_cost_df = production_df.merge(
    costs_slim, on=key, how="inner", validate="one_to_one"
)

site_columns = [c for c in sites_df.columns if c not in ["site_name", "commodity", "status"]]

integrated_df = production_cost_df.merge(
    sites_df[site_columns], on="site_id", how="left", validate="many_to_one"
)

print("Production rows:", len(production_df))
print("Cost rows:", len(costs_df))
print("Rows after production + cost merge:", len(production_cost_df))
print("Rows after site merge:", len(integrated_df))
print("Rows without matching site:", int(integrated_df["region"].isna().sum()))
print("Duplicate date-site keys:", int(integrated_df.duplicated(subset=key).sum()))
print("Columns:", integrated_df.shape[1])

assert len(integrated_df) == len(production_df) == len(costs_df)

Production rows: 1825
Cost rows: 1825
Rows after production + cost merge: 1825
Rows after site merge: 1825
Rows without matching site: 0
Duplicate date-site keys: 0
Columns: 37


## 5. Cost Reconciliation

`total_cost_aud` dibandingkan dengan jumlah tujuh komponen biaya. Selisih **dibulatkan ke sen** sebelum dihitung, karena membandingkan angka desimal mentah menghasilkan selisih semu akibat representasi floating-point (mis. 0,00999999 atau 4,7e-10).

In [8]:
cost_components = [
    "labor_cost_aud", "equipment_cost_aud", "fuel_cost_aud", "maintenance_cost_aud",
    "consumables_cost_aud", "utilities_cost_aud", "admin_cost_aud",
]

raw_difference = integrated_df["total_cost_aud"] - integrated_df[cost_components].sum(axis=1)
integrated_df["cost_difference_aud"] = raw_difference.round(2)

records = len(integrated_df)
abs_diff = integrated_df["cost_difference_aud"].abs()

cost_reconciliation = pd.DataFrame([
    {"metric": "Rows checked", "value": records},
    {"metric": "Exact matches (after rounding to cents)", "value": int((abs_diff == 0).sum())},
    {"metric": "Differences of AUD 0.01", "value": int((abs_diff == 0.01).sum())},
    {"metric": "Differences of AUD 0.02", "value": int((abs_diff == 0.02).sum())},
    {"metric": "Differences above AUD 0.02", "value": int((abs_diff > 0.02).sum())},
    {"metric": "Maximum absolute difference (AUD)", "value": float(abs_diff.max())},
    {"metric": "Raw differences above AUD 0.01 (floating-point artifact)", "value": int((raw_difference.abs() > 0.01).sum())},
])
display(cost_reconciliation)

display(
    integrated_df.assign(has_difference=abs_diff > 0)
    .groupby("site_id")["has_difference"]
    .agg(rows="count", rows_with_difference="sum")
    .reset_index()
)

,metric,value
0,Rows checked,1825.00
1,Exact matches (after rounding to cents),983.00
2,Differences of AUD 0.01,753.00
3,Differences of AUD 0.02,89.00
4,Differences above AUD 0.02,0.00
5,Maximum absolute difference (AUD),0.02
6,Raw differences above AUD 0.01 (floating-point artifact),506.00


,site_id,rows,rows_with_difference
0,COPP001,365,210
1,GOLD001,365,211
2,IRON001,365,216
3,LITH001,365,205
4,NICK001,365,0


**Interpretasi rekonsiliasi biaya**

- Dari 1.825 baris, 983 cocok persis, 753 berbeda A$0,01, dan 89 berbeda A$0,02. Tidak ada selisih di atas A$0,02.
- Versi sebelumnya melaporkan 617 baris cocok dan 1.208 berbeda. Angka itu timbul karena selisih desimal mentah tidak dibulatkan, bukan karena perbedaan nyata. Pada angka mentah, 506 baris juga tampak "di atas A$0,01" padahal tidak ada selisih di atas A$0,02 setelah dibulatkan.
- Selisih hanya muncul di empat lokasi yang berproduksi. Pola ini konsisten dengan pembulatan tiap komponen ke sen (lihat `01_Mining_Operations_Database_Audit.ipynb`).
- **Keputusan:** `total_cost_aud` tidak diubah. Selisih dibulatkan disimpan sebagai kolom QA `cost_difference_aud`.

## 6. Value and Business-Rule Checks

In [9]:
# Negative values in source numeric columns only.
# Derived columns are excluded, and so are coordinates (southern-hemisphere latitudes are legitimately negative).
source_numeric_columns = [
    col for col in integrated_df.select_dtypes(include="number").columns
    if col not in ["cost_difference_aud", "latitude", "longitude"]
]

negative_counts = {
    col: int((integrated_df[col] < 0).sum())
    for col in source_numeric_columns
    if (integrated_df[col] < 0).any()
}
print("Source columns with negative values:", negative_counts if negative_counts else "none")

hours_total = (
    integrated_df["operational_hours"] + integrated_df["downtime_hours"] + integrated_df["weather_delay_hours"]
)

business_rules = pd.DataFrame([
    {"rule": "tonnes_processed greater than tonnes_mined",
     "violating_rows": int((integrated_df["tonnes_processed"] > integrated_df["tonnes_mined"]).sum())},
    {"rule": "shift_1 + shift_2 differs from tonnes_mined",
     "violating_rows": int((integrated_df["shift_1_tonnes"] + integrated_df["shift_2_tonnes"] != integrated_df["tonnes_mined"]).sum())},
    {"rule": "recovery_rate outside 0-100",
     "violating_rows": int(((integrated_df["recovery_rate"] < 0) | (integrated_df["recovery_rate"] > 100)).sum())},
    {"rule": "total_cost_aud not positive",
     "violating_rows": int((integrated_df["total_cost_aud"] <= 0).sum())},
    {"rule": "operational + downtime + weather hours above 24",
     "violating_rows": int((hours_total > 24.01).sum())},
])
display(business_rules)

display(
    integrated_df.loc[hours_total > 24.01,
        ["date", "site_id", "operational_hours", "downtime_hours", "weather_delay_hours"]]
)

Source columns with negative values: none


,rule,violating_rows
0,tonnes_processed greater than tonnes_mined,0
1,shift_1 + shift_2 differs from tonnes_mined,0
2,recovery_rate outside 0-100,0
3,total_cost_aud not positive,0
4,operational + downtime + weather hours above 24,2


,date,site_id,operational_hours,downtime_hours,weather_delay_hours
751,2025-01-22,LITH001,12.0,12.0,12.0
799,2025-03-11,LITH001,16.8,7.2,7.2


**Interpretasi**

- Tidak ada nilai negatif pada kolom sumber (koordinat dikecualikan karena latitude negatif wajar di belahan bumi selatan), dan empat dari lima aturan bisnis tidak dilanggar.
- Dua baris Lithium Creek (22 Januari dan 11 Maret) memiliki total jam operasional + downtime + weather delay di atas 24 jam. Kemungkinan weather delay tercatat tumpang tindih dengan downtime. Baris tetap dipertahankan dan ditandai dengan kolom `flag_hours_over_24` pada Section 7.

### Zero-production records

In [10]:
zero_production = integrated_df[integrated_df["tonnes_mined"] == 0]

print("Rows with zero tonnes mined:", len(zero_production))

display(
    zero_production.groupby(["site_id", "status"]).agg(
        zero_production_days=("date", "count"),
        avg_total_cost_aud=("total_cost_aud", "mean"),
        reported_cost_per_unit=("cost_per_unit", "mean"),
    ).reset_index()
)

display(
    zero_production[["date", "site_id"] + cost_components + ["total_cost_aud"]].head(3)
)

Rows with zero tonnes mined: 365


,site_id,status,zero_production_days,avg_total_cost_aud,reported_cost_per_unit
0,NICK001,Care & Maintenance,365,8500.0,0.0


,date,site_id,labor_cost_aud,equipment_cost_aud,fuel_cost_aud,maintenance_cost_aud,consumables_cost_aud,utilities_cost_aud,admin_cost_aud,total_cost_aud
1460,2025-01-01,NICK001,5000.0,0.0,0.0,2000.0,0.0,1000.0,500.0,8500.0
1461,2025-01-02,NICK001,5000.0,0.0,0.0,2000.0,0.0,1000.0,500.0,8500.0
1462,2025-01-03,NICK001,5000.0,0.0,0.0,2000.0,0.0,1000.0,500.0,8500.0


**Interpretasi produksi nol**

- Ada 365 baris dengan produksi nol, semuanya milik Nickel Point Mine (*Care & Maintenance*). Baris ini dipertahankan karena merepresentasikan status operasi lokasi.
- Lokasi ini tetap mencatat biaya tetap A$8.500 per hari (labor 5.000, maintenance 2.000, utilities 1.000, admin 500). Karena tidak ada dokumentasi penyebabnya, ini dicatat sebagai biaya yang tetap ada selama *Care & Maintenance*, bukan kesalahan data.
- `cost_per_unit` pada baris ini tercatat 0, padahal biaya per unit tidak terdefinisi ketika produksi nol. Pada dataset analitis nilainya dijadikan kosong (`NaN`).

## 7. Feature Engineering

Fitur dibuat per kelompok dan seluruhnya didokumentasikan pada data dictionary (Section 10). Dataset berurutan per lokasi dan tanggal.

### 7.1 Calendar, site age, and automation label

In [11]:
df = integrated_df.copy()

df["date"] = pd.to_datetime(df["date"])
df["operating_since"] = pd.to_datetime(df["operating_since"])

df = df.sort_values(["site_id", "date"]).reset_index(drop=True)

# Each site must have one row for every consecutive day
date_gaps = df.groupby("site_id")["date"].apply(lambda s: s.diff().dropna().ne(pd.Timedelta(days=1)).sum())
assert (date_gaps == 0).all(), "Dates are not consecutive for every site."

# Calendar features
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["month_name"] = df["date"].dt.month_name()
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype(int)
df["day_of_week"] = df["date"].dt.day_name()
df["is_weekend"] = df["date"].dt.dayofweek >= 5

# Site age at the date of each record
df["site_age_years"] = ((df["date"] - df["operating_since"]).dt.days / 365.25).round(2)

# Automation level: keep the gap visible instead of guessing
df["automation_level_missing"] = df["automation_level"].isna()
df["automation_level"] = df["automation_level"].fillna("Unknown")

display(
    df[["date", "site_id", "year", "month", "quarter", "week_of_year", "day_of_week",
        "is_weekend", "site_age_years", "automation_level", "automation_level_missing"]].head(5)
)

,date,site_id,year,month,quarter,week_of_year,day_of_week,is_weekend,site_age_years,automation_level,automation_level_missing
0,2025-01-01,COPP001,2025,1,1,1,Wednesday,False,17.70,Low,False
1,2025-01-02,COPP001,2025,1,1,1,Thursday,False,17.71,Low,False
2,2025-01-03,COPP001,2025,1,1,1,Friday,False,17.71,Low,False
3,2025-01-04,COPP001,2025,1,1,1,Saturday,True,17.71,Low,False
4,2025-01-05,COPP001,2025,1,1,1,Sunday,True,17.71,Low,False


### 7.2 Production and cost features

In [12]:
# Production features
df["production_gap_tonnes"] = df["tonnes_mined"] - df["tonnes_processed"]
df["shift_1_share_pct"] = np.where(
    df["tonnes_mined"] > 0, df["shift_1_tonnes"] / df["tonnes_mined"] * 100, np.nan
)
df["has_downtime"] = df["downtime_hours"] > 0
df["has_weather_delay"] = df["weather_delay_hours"] > 0
df["flag_hours_over_24"] = (
    df["operational_hours"] + df["downtime_hours"] + df["weather_delay_hours"]
) > 24.01

# Cost features: undefined (NaN) when there is no production
df["analytical_cost_per_tonne_aud"] = np.where(
    df["tonnes_mined"] > 0, df["total_cost_aud"] / df["tonnes_mined"], np.nan
)
df["cost_per_tonne_processed_aud"] = np.where(
    df["tonnes_processed"] > 0, df["total_cost_aud"] / df["tonnes_processed"], np.nan
)

# Reported cost_per_unit: renamed, and undefined when there is no production.
# Its denominator differs by site (see the database audit), so it is not comparable across sites.
df = df.rename(columns={"cost_per_unit": "cost_per_unit_reported"})
df["cost_per_unit_reported"] = df["cost_per_unit_reported"].where(df["tonnes_mined"] > 0)

print("Rows with a valid cost per tonne mined:", int(df["analytical_cost_per_tonne_aud"].notna().sum()))
print("Rows with undefined cost per tonne mined:", int(df["analytical_cost_per_tonne_aud"].isna().sum()))

display(
    df.groupby("site_id")[["production_gap_tonnes", "shift_1_share_pct", "analytical_cost_per_tonne_aud",
                           "cost_per_tonne_processed_aud"]].mean().round(2).reset_index()
)

Rows with a valid cost per tonne mined: 1460
Rows with undefined cost per tonne mined: 365


,site_id,production_gap_tonnes,shift_1_share_pct,analytical_cost_per_tonne_aud,cost_per_tonne_processed_aud
0,COPP001,1.58,49.00,4078.68,4211.19
1,GOLD001,663.46,49.73,80.30,81.94
2,IRON001,2572.31,50.04,27.05,27.60
3,LITH001,36.30,49.79,638.72,651.94
4,NICK001,0.00,NaN,NaN,NaN


### 7.3 Lag and rolling features
Dihitung per lokasi dan hanya memakai hari **sebelumnya** (`shift(1)`), sehingga nilai hari ini tidak ikut terhitung dan tidak terjadi kebocoran data pada model prediksi. Tujuh hari pertama setiap lokasi kosong karena belum ada riwayat.

In [13]:
by_site = df.groupby("site_id")

df["tonnes_mined_lag_1"] = by_site["tonnes_mined"].shift(1)
df["tonnes_mined_roll7_prior_mean"] = by_site["tonnes_mined"].transform(
    lambda s: s.shift(1).rolling(7).mean()
)
df["cost_per_tonne_roll7_prior_mean"] = by_site["analytical_cost_per_tonne_aud"].transform(
    lambda s: s.shift(1).rolling(7).mean()
)

display(
    df.loc[df["site_id"] == "IRON001",
           ["date", "tonnes_mined", "tonnes_mined_lag_1", "tonnes_mined_roll7_prior_mean"]].head(10)
)

,date,tonnes_mined,tonnes_mined_lag_1,tonnes_mined_roll7_prior_mean
730,2025-01-01,113905,NaN,NaN
731,2025-01-02,127238,113905.0,NaN
732,2025-01-03,124194,127238.0,NaN
733,2025-01-04,117527,124194.0,NaN
734,2025-01-05,143865,117527.0,NaN
735,2025-01-06,140732,143865.0,NaN
736,2025-01-07,118896,140732.0,NaN
737,2025-01-08,125959,118896.0,126622.428571
738,2025-01-09,138138,125959.0,128344.428571
739,2025-01-10,138257,138138.0,129901.571429


### 7.4 Equipment features (site-day)
`equipment_performance` berisi satu baris per unit per hari, sehingga diagregasi ke level lokasi-hari sebelum digabung. Nickel Point tidak memiliki data peralatan, jadi nilainya kosong (bukan nol).

In [14]:
equipment_df = tables["equipment_performance"].copy()
equipment_df["date"] = pd.to_datetime(equipment_df["date"])

equipment_daily = (
    equipment_df.groupby(["date", "site_id"])
    .agg(
        equipment_units=("equipment_id", "nunique"),
        avg_oee_pct=("oee_pct", "mean"),
        avg_utilization_pct=("utilization_pct", "mean"),
        total_breakdown_hours=("breakdown_hours", "sum"),
        total_maintenance_hours=("maintenance_hours", "sum"),
        total_equipment_fuel_liters=("fuel_consumed_liters", "sum"),
        total_equipment_maintenance_cost_aud=("maintenance_cost_aud", "sum"),
        autonomous_unit_share=("autonomous", "mean"),
    )
    .reset_index()
)

for col in ["avg_oee_pct", "avg_utilization_pct", "total_breakdown_hours", "total_maintenance_hours",
            "total_equipment_maintenance_cost_aud"]:
    equipment_daily[col] = equipment_daily[col].round(2)
equipment_daily["autonomous_unit_share"] = equipment_daily["autonomous_unit_share"].round(3)

rows_before = len(df)
df = df.merge(equipment_daily, on=["date", "site_id"], how="left", validate="one_to_one")
assert len(df) == rows_before, "Equipment merge changed the number of rows."

df["equipment_units"] = df["equipment_units"].astype("Int64")
df["total_equipment_fuel_liters"] = df["total_equipment_fuel_liters"].astype("Int64")

print("Equipment site-day records:", len(equipment_daily))
print("Rows with equipment data:", int(df["equipment_units"].notna().sum()))
print("Rows without equipment data (no equipment records):", int(df["equipment_units"].isna().sum()))

display(
    df.groupby("site_id")[["equipment_units", "avg_oee_pct", "avg_utilization_pct",
                           "total_breakdown_hours", "autonomous_unit_share"]].mean().round(2).reset_index()
)

Equipment site-day records: 1460
Rows with equipment data: 1460
Rows without equipment data (no equipment records): 365


,site_id,equipment_units,avg_oee_pct,avg_utilization_pct,total_breakdown_hours,autonomous_unit_share
0,COPP001,5.0,61.00,62.12,0.83,0.0
1,GOLD001,7.0,77.92,79.28,1.13,0.0
2,IRON001,6.0,84.90,86.29,0.86,0.5
3,LITH001,4.0,79.06,80.27,0.47,0.0
4,NICK001,<NA>,NaN,NaN,NaN,NaN


### 7.5 Safety features (site-day)
Insiden dihitung per lokasi per hari. Hari tanpa insiden diberi nilai **0** hanya untuk lokasi yang memiliki catatan keselamatan. Nickel Point tidak memiliki catatan keselamatan sama sekali, sehingga nilainya dibiarkan kosong supaya "tidak ada data" tidak terbaca sebagai "tidak ada insiden".

In [15]:
incidents_df = tables["safety_incidents"].copy()
incidents_df["date"] = pd.to_datetime(incidents_df["date"])

incident_daily = (
    incidents_df.groupby(["date", "site_id"])
    .agg(
        incident_count=("incident_id", "count"),
        days_lost=("days_lost", "sum"),
        high_severity_incidents=("severity", lambda s: int((s == "High").sum())),
    )
    .reset_index()
)

rows_before = len(df)
df = df.merge(incident_daily, on=["date", "site_id"], how="left", validate="one_to_one")
assert len(df) == rows_before, "Safety merge changed the number of rows."

safety_columns = ["incident_count", "days_lost", "high_severity_incidents"]
safety_sites = tables["safety_metrics_summary"]["site_id"].unique()
covered = df["site_id"].isin(safety_sites)

for col in safety_columns:
    df.loc[covered, col] = df.loc[covered, col].fillna(0)
    df[col] = df[col].astype("Int64")

print("Sites with safety records:", sorted(safety_sites))
print("Site-days with at least one incident:", int((df["incident_count"] > 0).sum()))
print("Total incidents in prepared data:", int(df["incident_count"].sum()))
print("Total days lost in prepared data:", int(df["days_lost"].sum()))
print("Rows without safety records:", int(df["incident_count"].isna().sum()))

Sites with safety records: ['COPP001', 'GOLD001', 'IRON001', 'LITH001']
Site-days with at least one incident: 323
Total incidents in prepared data: 360
Total days lost in prepared data: 111
Rows without safety records: 365


## 8. Prepared Dataset and Validation

In [16]:
column_groups = {
    "Identity": ["date", "site_id", "site_name", "commodity", "status", "region", "mine_type"],
    "Site attributes": [
        "latitude", "longitude", "annual_target_tonnes", "daily_target_tonnes", "target_recovery_rate",
        "target_head_grade", "workforce_count", "automation_level", "automation_level_missing",
        "cost_per_unit_target", "operating_since", "site_age_years",
    ],
    "Production (source)": [
        "tonnes_mined", "tonnes_processed", "head_grade", "recovery_rate", "metal_produced",
        "plan_adherence_pct", "shift_1_tonnes", "shift_2_tonnes", "operational_hours",
        "downtime_hours", "weather_delay_hours",
    ],
    "Cost (source and QA)": cost_components + ["total_cost_aud", "cost_per_unit_reported", "cost_difference_aud"],
    "Calendar": ["year", "month", "month_name", "quarter", "week_of_year", "day_of_week", "is_weekend"],
    "Production features": [
        "production_gap_tonnes", "shift_1_share_pct", "has_downtime", "has_weather_delay", "flag_hours_over_24",
    ],
    "Cost features": ["analytical_cost_per_tonne_aud", "cost_per_tonne_processed_aud"],
    "Lag and rolling features": [
        "tonnes_mined_lag_1", "tonnes_mined_roll7_prior_mean", "cost_per_tonne_roll7_prior_mean",
    ],
    "Equipment features": [
        "equipment_units", "avg_oee_pct", "avg_utilization_pct", "total_breakdown_hours",
        "total_maintenance_hours", "total_equipment_fuel_liters",
        "total_equipment_maintenance_cost_aud", "autonomous_unit_share",
    ],
    "Safety features": safety_columns,
}

ordered_columns = [col for group in column_groups.values() for col in group]

# Every column must be placed in a group, and nothing may be left out by accident
unplaced = set(df.columns) - set(ordered_columns)
missing_from_df = set(ordered_columns) - set(df.columns)
print("Columns not placed in any group:", sorted(unplaced))
print("Grouped columns missing from the data:", sorted(missing_from_df))
assert not unplaced and not missing_from_df

prepared_df = df[ordered_columns].sort_values(["site_id", "date"]).reset_index(drop=True)

print("\nPrepared dataset shape:", prepared_df.shape)
print("Date range:", prepared_df["date"].min().date(), "to", prepared_df["date"].max().date())
print("Duplicate date-site keys:", int(prepared_df.duplicated(subset=["date", "site_id"]).sum()))
print("Unique mine sites:", prepared_df["site_id"].nunique())

display(pd.DataFrame({group: [len(cols)] for group, cols in column_groups.items()}, index=["columns"]).T)

Columns not placed in any group: []
Grouped columns missing from the data: []

Prepared dataset shape: (1825, 68)
Date range: 2025-01-01 to 2025-12-31
Duplicate date-site keys: 0
Unique mine sites: 5


,columns
Identity,7
Site attributes,12
Production (source),11
Cost (source and QA),10
Calendar,7
Production features,5
Cost features,2
Lag and rolling features,3
Equipment features,8
Safety features,3


### Missing values by column

In [17]:
missing_reasons = {
    "cost_per_unit_reported": "Undefined when tonnes mined is zero (Nickel Point)",
    "shift_1_share_pct": "Undefined when tonnes mined is zero (Nickel Point)",
    "analytical_cost_per_tonne_aud": "Undefined when tonnes mined is zero (Nickel Point)",
    "cost_per_tonne_processed_aud": "Undefined when tonnes processed is zero (Nickel Point)",
    "tonnes_mined_lag_1": "First day of each site has no previous day",
    "tonnes_mined_roll7_prior_mean": "First 7 days of each site have no full history",
    "cost_per_tonne_roll7_prior_mean": "No full history, and undefined for Nickel Point",
    "equipment_units": "No equipment records for Nickel Point",
    "avg_oee_pct": "No equipment records for Nickel Point",
    "avg_utilization_pct": "No equipment records for Nickel Point",
    "total_breakdown_hours": "No equipment records for Nickel Point",
    "total_maintenance_hours": "No equipment records for Nickel Point",
    "total_equipment_fuel_liters": "No equipment records for Nickel Point",
    "total_equipment_maintenance_cost_aud": "No equipment records for Nickel Point",
    "autonomous_unit_share": "No equipment records for Nickel Point",
    "incident_count": "No safety records for Nickel Point",
    "days_lost": "No safety records for Nickel Point",
    "high_severity_incidents": "No safety records for Nickel Point",
}

missing_counts = prepared_df.isna().sum()
missing_counts = missing_counts[missing_counts > 0]

unexplained = set(missing_counts.index) - set(missing_reasons)
assert not unexplained, f"Missing values without a documented reason: {sorted(unexplained)}"

# Outside the time-series features, every missing value must belong to Nickel Point
time_series_columns = ["tonnes_mined_lag_1", "tonnes_mined_roll7_prior_mean", "cost_per_tonne_roll7_prior_mean"]
operating_rows = prepared_df["site_id"] != "NICK001"
for col in missing_counts.index:
    if col not in time_series_columns:
        assert prepared_df.loc[operating_rows, col].isna().sum() == 0, f"{col} has missing values outside Nickel Point"

missing_summary = (
    missing_counts.rename("missing_count").to_frame()
    .assign(reason=lambda t: t.index.map(missing_reasons))
    .sort_values("missing_count", ascending=False)
)
display(missing_summary)

,missing_count,reason
cost_per_tonne_roll7_prior_mean,393,"No full history, and undefined for Nickel Point"
cost_per_unit_reported,365,Undefined when tonnes mined is zero (Nickel Point)
shift_1_share_pct,365,Undefined when tonnes mined is zero (Nickel Point)
analytical_cost_per_tonne_aud,365,Undefined when tonnes mined is zero (Nickel Point)
equipment_units,365,No equipment records for Nickel Point
cost_per_tonne_processed_aud,365,Undefined when tonnes processed is zero (Nickel Point)
avg_oee_pct,365,No equipment records for Nickel Point
avg_utilization_pct,365,No equipment records for Nickel Point
total_maintenance_hours,365,No equipment records for Nickel Point
total_breakdown_hours,365,No equipment records for Nickel Point


**Interpretasi nilai kosong**

- Semua nilai kosong punya alasan yang terdokumentasi, dan di luar fitur lag/rolling seluruhnya milik Nickel Point (365 baris): biaya per ton tidak terdefinisi tanpa produksi, serta tidak ada data peralatan dan keselamatan.
- Fitur lag dan rolling kosong pada hari-hari awal tiap lokasi karena belum ada riwayat.
- `automation_level` tidak lagi kosong karena diberi label `Unknown` dengan penanda `automation_level_missing`.

### Reconciliation with the source database

In [18]:
def sql_value(query):
    return pd.read_sql_query(query, conn).iloc[0, 0]

reconciliation = pd.DataFrame([
    {"metric": "Total tonnes mined",
     "database": sql_value("SELECT SUM(tonnes_mined) FROM daily_production"),
     "prepared": prepared_df["tonnes_mined"].sum()},
    {"metric": "Total tonnes processed",
     "database": sql_value("SELECT SUM(tonnes_processed) FROM daily_production"),
     "prepared": prepared_df["tonnes_processed"].sum()},
    {"metric": "Total cost (AUD)",
     "database": sql_value("SELECT SUM(total_cost_aud) FROM daily_costs"),
     "prepared": prepared_df["total_cost_aud"].sum()},
    {"metric": "Total fuel consumed by equipment (liters)",
     "database": sql_value("SELECT SUM(fuel_consumed_liters) FROM equipment_performance"),
     "prepared": prepared_df["total_equipment_fuel_liters"].sum()},
    {"metric": "Total equipment maintenance cost (AUD)",
     "database": sql_value("SELECT SUM(maintenance_cost_aud) FROM equipment_performance"),
     "prepared": prepared_df["total_equipment_maintenance_cost_aud"].sum()},
    {"metric": "Total safety incidents",
     "database": sql_value("SELECT COUNT(*) FROM safety_incidents"),
     "prepared": prepared_df["incident_count"].sum()},
    {"metric": "Total days lost",
     "database": sql_value("SELECT SUM(days_lost) FROM safety_incidents"),
     "prepared": prepared_df["days_lost"].sum()},
])

reconciliation["match"] = np.isclose(
    reconciliation["database"].astype(float), reconciliation["prepared"].astype(float), rtol=1e-6, atol=0.5
)
display(reconciliation)

assert reconciliation["match"].all(), "Prepared dataset does not reconcile with the database."

,metric,database,prepared,match
0,Total tonnes mined,5.970681e+07,5.970681e+07,True
1,Total tonnes processed,5.851193e+07,5.851193e+07,True
2,Total cost (AUD),2.736541e+09,2.736541e+09,True
3,Total fuel consumed by equipment (liters),2.977249e+07,2.977249e+07,True
4,Total equipment maintenance cost (AUD),2.149575e+07,2.149575e+07,True
5,Total safety incidents,3.600000e+02,3.600000e+02,True
6,Total days lost,1.110000e+02,1.110000e+02,True


**Interpretasi rekonsiliasi**

Total tonase, biaya, bahan bakar, biaya perawatan peralatan, insiden, dan hari hilang pada dataset analitis sama dengan total pada database. Agregasi ke level lokasi-hari tidak menghilangkan atau menggandakan data.

### Data types and constant columns

In [19]:
dtype_summary = (
    prepared_df.dtypes.astype(str).value_counts().rename_axis("dtype").reset_index(name="columns")
)
display(dtype_summary)

constant_columns = [col for col in prepared_df.columns if prepared_df[col].nunique(dropna=True) <= 1]
print("Columns with a single value:", constant_columns)

,dtype,columns
0,float64,35
1,str,9
2,int64,9
3,bool,5
4,Int64,5
5,int32,3
6,datetime64[us],2


Columns with a single value: ['year']


Kolom `year` hanya berisi satu nilai (2025) karena data mencakup satu tahun. Kolom ini dipertahankan agar dataset tetap bisa digabung dengan data tahun lain, tetapi tidak berguna sebagai fitur model.

## 9. Notes for Downstream Use

Beberapa kolom berhubungan secara deterministik atau hampir deterministik. Hubungan ini dihitung di bawah supaya tidak terjadi kebocoran data atau multikolinearitas saat membuat model.

In [20]:
producing = prepared_df[prepared_df["tonnes_mined"] > 0].copy()

# 1) Cost components as a share of total cost (producing sites)
component_shares = producing[cost_components].div(producing["total_cost_aud"], axis=0)
share_check = pd.DataFrame({
    "mean_share": component_shares.mean().round(4),
    "spread_within_data": (component_shares.max() - component_shares.min()).round(6),
})
display(share_check)

# 2) plan_adherence_pct vs tonnes_mined / daily_target_tonnes
implied_plan_adherence = producing["tonnes_mined"] / producing["daily_target_tonnes"] * 100
print(
    "plan_adherence_pct vs tonnes_mined / daily target: correlation =",
    round(producing["plan_adherence_pct"].corr(implied_plan_adherence), 3),
    "| mean absolute difference (pts) =",
    round((producing["plan_adherence_pct"] - implied_plan_adherence).abs().mean(), 2),
)

# 3) Processed share of mined tonnes
processing_ratio = producing["tonnes_processed"] / producing["tonnes_mined"] * 100
print("tonnes_processed as % of tonnes_mined: median =", round(processing_ratio.median(), 2),
      "| 5th-95th percentile =", round(processing_ratio.quantile(0.05), 2), "to", round(processing_ratio.quantile(0.95), 2))

# 4) Scale differences between sites
display(
    producing.groupby("site_name")["tonnes_mined"].agg(avg_daily_tonnes_mined="mean").round(0).reset_index()
)

,mean_share,spread_within_data
labor_cost_aud,0.40,0.0
equipment_cost_aud,0.28,0.0
fuel_cost_aud,0.18,0.0
maintenance_cost_aud,0.08,0.0
consumables_cost_aud,0.04,0.0
utilities_cost_aud,0.01,0.0
admin_cost_aud,0.01,0.0


plan_adherence_pct vs tonnes_mined / daily target: correlation = 1.0 | mean absolute difference (pts) = 0.07
tonnes_processed as % of tonnes_mined: median = 98.0 | 5th-95th percentile = 96.15 to 98.0


,site_name,avg_daily_tonnes_mined
0,Copper Hills Mine,51.0
1,Golden Valley Mine,33149.0
2,Iron Ridge Mine,128590.0
3,Lithium Creek Mine,1791.0


**Catatan untuk analisis dan model**

- **Komponen biaya = porsi tetap dari total.** Setiap komponen adalah persentase konstan dari `total_cost_aud` (labor 40%, equipment 28%, fuel 18%, maintenance 8%, consumables 4%, utilities 1%, admin 1%). Kolom-kolom ini kolinear sempurna dengan total biaya, jadi jangan dipakai bersama sebagai prediktor biaya, dan komposisi biaya tidak mencerminkan struktur biaya tambang nyata.
- **`plan_adherence_pct` hampir identik dengan `tonnes_mined` ÷ `daily_target_tonnes`.** Jangan memakainya sebagai prediktor `tonnes_mined` (kebocoran data).
- **`tonnes_processed` sekitar 98% dari `tonnes_mined`**, sehingga keduanya nyaris duplikat sebagai prediktor.
- **Skala antar lokasi sangat berbeda.** Rata-rata produksi harian Iron Ridge sekitar 128 ribu ton, sedangkan Copper Hills sekitar 50 ton. Model gabungan akan didominasi Iron Ridge, jadi pertimbangkan model per lokasi atau normalisasi per lokasi.
- **Nickel Point** (produksi nol) sebaiknya dikeluarkan dari model produksi dan biaya per ton, tetapi tetap berguna untuk analisis status.
- **OEE = availability × utilization** (temuan audit), sehingga `avg_oee_pct` dan `avg_utilization_pct` saling berkaitan erat.
- **Hanya satu tahun data.** Pola musiman tidak bisa divalidasi, dan evaluasi model prediksi sebaiknya memakai pembagian berdasarkan waktu, bukan acak.
- `cost_per_unit_reported` memakai pembagi berbeda per lokasi, sehingga tidak boleh dibandingkan antar lokasi.

## 10. Data Dictionary and Export

In [21]:
dictionary_entries = {
    # Identity
    "date": ("Source: daily_production", "Calendar date of the record (datetime)."),
    "site_id": ("Source: daily_production", "Mine site identifier. Key together with date."),
    "site_name": ("Source: daily_production", "Mine site name (identical across all source tables)."),
    "commodity": ("Source: daily_production", "Commodity produced at the site."),
    "status": ("Source: daily_production", "Operating status of the site (Operational or Care & Maintenance)."),
    "region": ("Source: mine_sites", "Geographic region of the site."),
    "mine_type": ("Source: mine_sites", "Open pit, underground, or combined."),
    # Site attributes
    "latitude": ("Source: mine_sites", "Site latitude."),
    "longitude": ("Source: mine_sites", "Site longitude."),
    "annual_target_tonnes": ("Source: mine_sites", "Annual production target in tonnes."),
    "daily_target_tonnes": ("Source: mine_sites", "Daily production target in tonnes."),
    "target_recovery_rate": ("Source: mine_sites", "Target recovery rate (%)."),
    "target_head_grade": ("Source: mine_sites", "Target head grade."),
    "workforce_count": ("Source: mine_sites", "Number of workers at the site."),
    "automation_level": ("Source: mine_sites, cleaned", "Automation level (Low, Medium, High). Unknown for Nickel Point, where the source value is empty."),
    "automation_level_missing": ("Derived", "True when the source automation_level was empty."),
    "cost_per_unit_target": ("Source: mine_sites", "Target cost per unit. The unit basis differs by commodity."),
    "operating_since": ("Source: mine_sites", "Date the site started operating (datetime)."),
    "site_age_years": ("Derived", "Years between operating_since and the record date."),
    # Production
    "tonnes_mined": ("Source: daily_production", "Tonnes mined on the day. Identical to the cost table's tonnes_produced, which was dropped."),
    "tonnes_processed": ("Source: daily_production", "Tonnes processed on the day."),
    "head_grade": ("Source: daily_production", "Average grade of the ore processed."),
    "recovery_rate": ("Source: daily_production", "Recovery rate (%)."),
    "metal_produced": ("Source: daily_production", "Metal produced. The unit depends on the commodity and is not documented in the source."),
    "plan_adherence_pct": ("Source: daily_production", "Production as % of the daily target. Nearly identical to tonnes_mined / daily_target_tonnes."),
    "shift_1_tonnes": ("Source: daily_production", "Tonnes mined in shift 1."),
    "shift_2_tonnes": ("Source: daily_production", "Tonnes mined in shift 2."),
    "operational_hours": ("Source: daily_production", "Hours of operation on the day."),
    "downtime_hours": ("Source: daily_production", "Hours of downtime on the day."),
    "weather_delay_hours": ("Source: daily_production", "Hours lost to weather delay."),
    # Cost
    "labor_cost_aud": ("Source: daily_costs", "Labor cost (AUD)."),
    "equipment_cost_aud": ("Source: daily_costs", "Equipment cost (AUD)."),
    "fuel_cost_aud": ("Source: daily_costs", "Fuel cost (AUD)."),
    "maintenance_cost_aud": ("Source: daily_costs", "Site maintenance cost (AUD)."),
    "consumables_cost_aud": ("Source: daily_costs", "Consumables cost (AUD)."),
    "utilities_cost_aud": ("Source: daily_costs", "Utilities cost (AUD)."),
    "admin_cost_aud": ("Source: daily_costs", "Administration cost (AUD)."),
    "total_cost_aud": ("Source: daily_costs", "Recorded total cost (AUD). Not adjusted."),
    "cost_per_unit_reported": ("Source: daily_costs, renamed", "Reported cost per unit. The denominator differs by site (tonnes, metal unit, or pound-equivalent). Empty when there is no production (reported as 0 in the source)."),
    "cost_difference_aud": ("Derived (QA)", "total_cost_aud minus the sum of the seven components, rounded to cents."),
    # Calendar
    "year": ("Derived", "Calendar year. Constant (2025) in this dataset."),
    "month": ("Derived", "Month number (1-12)."),
    "month_name": ("Derived", "Month name."),
    "quarter": ("Derived", "Calendar quarter (1-4)."),
    "week_of_year": ("Derived", "ISO week number."),
    "day_of_week": ("Derived", "Day name."),
    "is_weekend": ("Derived", "True on Saturday and Sunday."),
    # Production features
    "production_gap_tonnes": ("Derived", "tonnes_mined minus tonnes_processed."),
    "shift_1_share_pct": ("Derived", "Share of daily tonnes mined in shift 1 (%). Empty when no tonnes were mined."),
    "has_downtime": ("Derived", "True when downtime_hours is above 0."),
    "has_weather_delay": ("Derived", "True when weather_delay_hours is above 0."),
    "flag_hours_over_24": ("Derived (QA)", "True when operational + downtime + weather delay hours exceed 24."),
    # Cost features
    "analytical_cost_per_tonne_aud": ("Derived", "total_cost_aud / tonnes_mined. Empty when no tonnes were mined."),
    "cost_per_tonne_processed_aud": ("Derived", "total_cost_aud / tonnes_processed. Empty when nothing was processed."),
    # Lag and rolling
    "tonnes_mined_lag_1": ("Derived", "tonnes_mined of the previous day for the same site."),
    "tonnes_mined_roll7_prior_mean": ("Derived", "Mean tonnes_mined over the 7 days before the record date (excludes the current day)."),
    "cost_per_tonne_roll7_prior_mean": ("Derived", "Mean analytical cost per tonne over the 7 days before the record date (excludes the current day)."),
    # Equipment
    "equipment_units": ("Derived: equipment_performance", "Number of equipment units recorded at the site on the day."),
    "avg_oee_pct": ("Derived: equipment_performance", "Mean OEE across units (%)."),
    "avg_utilization_pct": ("Derived: equipment_performance", "Mean utilization across units (%)."),
    "total_breakdown_hours": ("Derived: equipment_performance", "Sum of breakdown hours across units."),
    "total_maintenance_hours": ("Derived: equipment_performance", "Sum of maintenance hours across units."),
    "total_equipment_fuel_liters": ("Derived: equipment_performance", "Sum of fuel consumed by the tracked units (liters)."),
    "total_equipment_maintenance_cost_aud": ("Derived: equipment_performance", "Sum of maintenance cost of the tracked units (AUD). Covers only about 10% of site maintenance cost."),
    "autonomous_unit_share": ("Derived: equipment_performance", "Share of tracked units that are autonomous (0-1)."),
    # Safety
    "incident_count": ("Derived: safety_incidents", "Number of safety incidents on the day. Empty for sites without safety records."),
    "days_lost": ("Derived: safety_incidents", "Days lost from incidents on the day."),
    "high_severity_incidents": ("Derived: safety_incidents", "Number of High-severity incidents on the day."),
}

group_of = {col: group for group, cols in column_groups.items() for col in cols}

data_dictionary = pd.DataFrame(
    [
        {
            "column": col,
            "group": group_of[col],
            "dtype": str(prepared_df[col].dtype),
            "source": dictionary_entries[col][0],
            "description": dictionary_entries[col][1],
            "missing_values": int(prepared_df[col].isna().sum()),
        }
        for col in prepared_df.columns
    ]
)

assert set(dictionary_entries) == set(prepared_df.columns), "Data dictionary does not match the dataset columns."

display(data_dictionary)

,column,group,dtype,source,description,missing_values
0,date,Identity,datetime64[us],Source: daily_production,Calendar date of the record (datetime).,0
1,site_id,Identity,str,Source: daily_production,Mine site identifier. Key together with date.,0
2,site_name,Identity,str,Source: daily_production,Mine site name (identical across all source tables).,0
3,commodity,Identity,str,Source: daily_production,Commodity produced at the site.,0
4,status,Identity,str,Source: daily_production,Operating status of the site (Operational or Care & Maintenance).,0
...,...,...,...,...,...,...
63,total_equipment_maintenance_cost_aud,Equipment features,float64,Derived: equipment_performance,Sum of maintenance cost of the tracked units (AUD). Covers only about 10% of site maintenance cost.,365
64,autonomous_unit_share,Equipment features,float64,Derived: equipment_performance,Share of tracked units that are autonomous (0-1).,365
65,incident_count,Safety features,Int64,Derived: safety_incidents,Number of safety incidents on the day. Empty for sites without safety records.,365
66,days_lost,Safety features,Int64,Derived: safety_incidents,Days lost from incidents on the day.,365


### Export

In [22]:
dataset_path = OUTPUT_DIR / "mining_operations_prepared.csv"
dictionary_path = OUTPUT_DIR / "mining_operations_data_dictionary.csv"

prepared_df.to_csv(dataset_path, index=False)
data_dictionary.to_csv(dictionary_path, index=False)

print("Dataset exported:", dataset_path)
print("Data dictionary exported:", dictionary_path)
print("Rows exported:", len(prepared_df))
print("Columns exported:", len(prepared_df.columns))
print("File size (KB):", round(dataset_path.stat().st_size / 1024, 1))

Dataset exported: /content/drive/MyDrive/Mining-Operation/mining_operations_prepared.csv
Data dictionary exported: /content/drive/MyDrive/Mining-Operation/mining_operations_data_dictionary.csv
Rows exported: 1825
Columns exported: 68
File size (KB): 819.5


### Export verification

In [23]:
check_df = pd.read_csv(dataset_path, parse_dates=["date", "operating_since"])

verification = pd.DataFrame([
    {"check": "Same shape as prepared dataset", "passed": check_df.shape == prepared_df.shape},
    {"check": "Same column names and order", "passed": list(check_df.columns) == list(prepared_df.columns)},
    {"check": "No duplicate date-site keys", "passed": int(check_df.duplicated(subset=["date", "site_id"]).sum()) == 0},
    {"check": "Same missing values per column",
     "passed": bool((check_df.isna().sum() == prepared_df.isna().sum()).all())},
    {"check": "Numeric totals preserved",
     "passed": bool(np.allclose(
         check_df.select_dtypes("number").sum(), prepared_df.select_dtypes("number").sum().astype(float), rtol=1e-9, atol=1e-6))},
])
display(verification)

assert verification["passed"].all(), "Exported CSV does not match the prepared dataset."

display(check_df.head(5))

,check,passed
0,Same shape as prepared dataset,True
1,Same column names and order,True
2,No duplicate date-site keys,True
3,Same missing values per column,True
4,Numeric totals preserved,True


,date,site_id,site_name,commodity,status,region,mine_type,latitude,longitude,annual_target_tonnes,daily_target_tonnes,target_recovery_rate,target_head_grade,workforce_count,automation_level,automation_level_missing,cost_per_unit_target,operating_since,site_age_years,tonnes_mined,tonnes_processed,head_grade,recovery_rate,metal_produced,plan_adherence_pct,shift_1_tonnes,shift_2_tonnes,operational_hours,downtime_hours,weather_delay_hours,labor_cost_aud,equipment_cost_aud,fuel_cost_aud,maintenance_cost_aud,consumables_cost_aud,utilities_cost_aud,admin_cost_aud,total_cost_aud,cost_per_unit_reported,cost_difference_aud,year,month,month_name,quarter,week_of_year,day_of_week,is_weekend,production_gap_tonnes,shift_1_share_pct,has_downtime,has_weather_delay,flag_hours_over_24,analytical_cost_per_tonne_aud,cost_per_tonne_processed_aud,tonnes_mined_lag_1,tonnes_mined_roll7_prior_mean,cost_per_tonne_roll7_prior_mean,equipment_units,avg_oee_pct,avg_utilization_pct,total_breakdown_hours,total_maintenance_hours,total_equipment_fuel_liters,total_equipment_maintenance_cost_aud,autonomous_unit_share,incident_count,days_lost,high_severity_incidents
0,2025-01-01,COPP001,Copper Hills Mine,Copper,Operational,Mid West,Underground,-28.4521,117.1234,20000,55,88.0,2.8,280,Low,False,1.85,2007-04-20,17.70,48,47,2.92,88.2,1.21,87.3,23,25,24.0,0.0,0.0,76028.60,53220.02,34212.87,15205.72,7602.86,1900.72,1900.72,190071.51,71.40,0.00,2025,1,January,1,1,Wednesday,False,1,47.916667,False,False,False,3959.823125,4044.074681,NaN,NaN,NaN,5.0,64.46,64.46,0.0,0.0,5709.0,7122.52,0.0,0.0,0.0,0.0
1,2025-01-02,COPP001,Copper Hills Mine,Copper,Operational,Mid West,Underground,-28.4521,117.1234,20000,55,88.0,2.8,280,Low,False,1.85,2007-04-20,17.71,50,49,2.85,86.0,1.20,90.9,25,25,24.0,0.0,0.0,84099.36,58869.56,37844.71,16819.87,8409.94,2102.48,2102.48,210248.41,79.64,0.01,2025,1,January,1,1,Thursday,False,1,50.000000,False,False,False,4204.968200,4290.783878,48.0,NaN,NaN,5.0,60.18,60.18,0.0,0.0,5295.0,6976.65,0.0,0.0,0.0,0.0
2,2025-01-03,COPP001,Copper Hills Mine,Copper,Operational,Mid West,Underground,-28.4521,117.1234,20000,55,88.0,2.8,280,Low,False,1.85,2007-04-20,17.71,50,49,2.93,85.4,1.23,90.9,25,25,24.0,0.0,0.0,83317.02,58321.92,37492.66,16663.40,8331.70,2082.93,2082.93,208292.55,76.97,-0.01,2025,1,January,1,1,Friday,False,1,50.000000,False,False,False,4165.851000,4250.868367,50.0,NaN,NaN,5.0,62.02,62.02,0.0,0.0,5472.0,6193.37,0.0,0.0,0.0,0.0
3,2025-01-04,COPP001,Copper Hills Mine,Copper,Operational,Mid West,Underground,-28.4521,117.1234,20000,55,88.0,2.8,280,Low,False,1.85,2007-04-20,17.71,57,55,2.85,87.7,1.37,103.6,30,27,24.0,0.0,0.0,90721.35,63504.94,40824.61,18144.27,9072.13,2268.03,2268.03,226803.37,75.25,0.01,2025,1,January,1,1,Saturday,True,2,52.631579,False,False,False,3979.006491,4123.697636,50.0,NaN,NaN,5.0,64.48,64.48,0.0,0.0,5709.0,5465.68,0.0,0.0,0.0,0.0
4,2025-01-05,COPP001,Copper Hills Mine,Copper,Operational,Mid West,Underground,-28.4521,117.1234,20000,55,88.0,2.8,280,Low,False,1.85,2007-04-20,17.71,45,44,2.75,85.8,1.04,81.8,24,21,24.0,0.0,0.0,75116.14,52581.30,33802.26,15023.23,7511.61,1877.90,1877.90,187790.34,82.08,0.00,2025,1,January,1,1,Sunday,True,1,53.333333,False,False,False,4173.118667,4267.962273,57.0,NaN,NaN,5.0,62.22,62.22,0.0,0.0,5492.0,6584.68,0.0,0.0,0.0,0.0


## 11. Summary and Conclusion

In [24]:
summary = {
    "dataset_name": "Mining Operations Prepared Dataset",
    "source_database": DB_PATH.name,
    "grain": "one row per mine site per day",
    "output_file": dataset_path.name,
    "data_dictionary_file": dictionary_path.name,
    "total_rows": len(prepared_df),
    "total_columns": len(prepared_df.columns),
    "base_columns": int(sum(len(column_groups[g]) for g in ["Identity", "Site attributes", "Production (source)", "Cost (source and QA)"])),
    "feature_columns": int(sum(len(column_groups[g]) for g in list(column_groups)[4:])),
    "unique_mine_sites": prepared_df["site_id"].nunique(),
    "duplicate_date_site_keys": int(prepared_df.duplicated(subset=["date", "site_id"]).sum()),
    "columns_with_missing_values": int((prepared_df.isna().sum() > 0).sum()),
    "date_start": prepared_df["date"].min().strftime("%Y-%m-%d"),
    "date_end": prepared_df["date"].max().strftime("%Y-%m-%d"),
}

summary_df = pd.DataFrame(list(summary.items()), columns=["metric", "value"])
display(summary_df)

,metric,value
0,dataset_name,Mining Operations Prepared Dataset
1,source_database,mining_operations.db
2,grain,one row per mine site per day
3,output_file,mining_operations_prepared.csv
4,data_dictionary_file,mining_operations_data_dictionary.csv
5,total_rows,1825
6,total_columns,68
7,base_columns,40
8,feature_columns,28
9,unique_mine_sites,5


## Project Summary

This notebook prepared the synthetic Mining Operations dataset for downstream analysis and machine learning. Production, cost, and mine-site data were integrated, equipment and safety data were aggregated to the same site-day grain, and the result was validated against the source database and exported with a data dictionary.

## Key Findings

- **Integration:** The prepared dataset has one row per mine site per day (1,825 rows, 5 sites, 1 January to 31 December 2025), with every source total reconciled to the database.
- **Redundant columns removed:** Duplicated `site_name`, `commodity`, and `status` columns were reduced to one copy each. `tonnes_produced` was dropped because it is identical to `tonnes_mined`.
- **Cost consistency:** After rounding to cents, `total_cost_aud` differs from the sum of its components by at most AUD 0.02 (983 exact matches, 842 rows off by one or two cents). The earlier count of 1,208 differing rows was a floating-point artifact. Source totals were not adjusted.
- **Zero production:** All 365 zero-production rows belong to Nickel Point Mine (*Care & Maintenance*), which still records AUD 8,500 of fixed daily cost. Cost-per-tonne and `cost_per_unit_reported` are left empty for these rows rather than recorded as 0.
- **Missing values:** `automation_level` is labeled `Unknown` for Nickel Point with a flag column, not guessed. All other missing values have a documented reason.
- **Added features:** Calendar and site-age features, production and cost ratios, lagged and rolling features that use only prior days, and site-day equipment and safety features.
- **Data quality flag:** Two Lithium Creek days have daily hours above 24 and are flagged with `flag_hours_over_24`.

## Cautions for Modeling

- Cost components are fixed shares of total cost and perfectly collinear with it.
- `plan_adherence_pct` is nearly identical to `tonnes_mined` divided by the daily target, so using it to predict tonnes mined leaks the target.
- Site scale differs enormously (Iron Ridge about 128,000 tonnes per day, Copper Hills about 50), so pooled models are dominated by Iron Ridge.
- Only one year of data is available, so seasonality cannot be validated and time-based splits are recommended.

## Output

`mining_operations_prepared.csv` and `mining_operations_data_dictionary.csv`, both saved next to the database and verified by reading them back.

**Data note:** This project uses a synthetic mining operations dataset for portfolio and analytical demonstration purposes. It does not represent actual operational data from a mining company.